## Exercise 1: Feature Extraction
**Dataset Used:** CIFAR-10 (Cats vs Dogs Subset)

1. Load MobileNetV2 (trainable=False). Add GlobalAveragePooling2D and Dense.
2. Train on 1000 images (500 cats, 500 dogs) for 5 epochs. Print Accuracy/Loss.
3. Confusion matrix. Which animal has more errors?

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Input
from tensorflow.keras.models import Model
from tensorflow.keras.datasets import cifar10
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

# Load CIFAR-10 and filter Cats(3) and Dogs(5)
(x_train, y_train), (x_test, y_test) = cifar10.load_data()
cats_train_idx = np.where(y_train == 3)[0][:500]
dogs_train_idx = np.where(y_train == 5)[0][:500]
train_idx = np.concatenate([cats_train_idx, dogs_train_idx])

cats_test_idx = np.where(y_test == 3)[0][:200]
dogs_test_idx = np.where(y_test == 5)[0][:200]
test_idx = np.concatenate([cats_test_idx, dogs_test_idx])

x_train_sub = x_train[train_idx].astype('float32') / 255.
y_train_sub = np.concatenate([np.zeros(500), np.ones(500)]) # 0: Cat, 1: Dog

x_test_sub = x_test[test_idx].astype('float32') / 255.
y_test_sub = np.concatenate([np.zeros(200), np.ones(200)])

# Resize for MobileNetV2 (requires min 32x32, CIFAR is 32x32, but works better larger. We will use 32x32 to be fast)
inputs = Input(shape=(32, 32, 3))
base_model = MobileNetV2(input_shape=(32, 32, 3), include_top=False, weights='imagenet')
base_model.trainable = False # Freeze base layers

x = base_model(inputs, training=False)
x = GlobalAveragePooling2D()(x)
outputs = Dense(1, activation='sigmoid')(x)
model = Model(inputs, outputs)

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Train
history = model.fit(x_train_sub, y_train_sub, epochs=1, validation_data=(x_test_sub, y_test_sub))

# Confusion Matrix
y_pred = (model.predict(x_test_sub) > 0.5).astype(int).reshape(-1)
cm = confusion_matrix(y_test_sub, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Cat', 'Dog'])
disp.plot(cmap='Blues')
plt.title("Confusion Matrix")
plt.show()

print("Base layers are frozen so we don't destroy the valuable pre-trained weights from ImageNet during the initial training phase with large gradients from our untrained head.")

x_train = x_train[:500]; y_train = y_train[:500]
x_test = x_test[:100]; y_test = y_test[:100]
